<a href="https://colab.research.google.com/github/emajebiCOS101/student-performance-analysis/blob/main/Eiza_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 4. Eiza - Model training and Comparison

This notebook contains my model training contribution to the group project. Its compares Logistic Regression and a Decision Making Tree using five fold stratified cross-validation, selects the model with the higher average validation F1 score, and trains it on the training set.

This section depends on x train and y trai from Solomon's data preparation section, The complete workflow can be ran in the Student_Performance_Project.ipynb file.

In [ ]:
# Importing models and training tools
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
# Creating a decision tree with limited depth
decision_tree = DecisionTreeClassifier(
    max_depth = 3,
    random_state = 42
)

In [ ]:
# Scaling features and applying logistic regression
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [ ]:
# Dividing the training data into five stratified folds
cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

In [ ]:
# Setting the metrics for comparing the models
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

In [ ]:
# Evaluating logistic regression using training data only
logistic_scores = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

# Evaluating the decision tree using the same folds
tree_scores = cross_validate(
    decision_tree,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

In [ ]:
# Displaying average training and validation scores
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Training F1": [
        logistic_scores["train_f1"].mean(),
        tree_scores["train_f1"].mean()
    ],
    "Validation Accuracy": [
        logistic_scores["test_accuracy"].mean(),
        tree_scores["test_accuracy"].mean()
    ],
    "Validation Precision": [
        logistic_scores["test_precision"].mean(),
        tree_scores["test_precision"].mean()
    ],
    "Validation Recall": [
        logistic_scores["test_recall"].mean(),
        tree_scores["test_recall"].mean()
    ],
    "Validation F1": [
        logistic_scores["test_f1"].mean(),
        tree_scores["test_f1"].mean()
    ]
})

comparison.round(3)

In [ ]:
# Selecting the model with the higher average validation F1
if logistic_scores["test_f1"].mean() >= tree_scores["test_f1"].mean():
    best_model = logistic_model
    best_model_name = "Logistic Regression"
else:
    best_model = decision_tree
    best_model_name = "Decision Tree"

# Training the selected model on all training records
best_model.fit(X_train, y_train)

print("Selected model:", best_model_name)